# Event and Interactive Tables

DO NOT RUN THIS NOTEBOOK ON CORPORATE ACCOUNT. IN FREE ACCOUNT, YOU MUST REMOVE TABLES TO AVOID ADDITIONAL COST.

Event tables capture operational telemetry. Interactive tables support selective analytical serving. This notebook includes separate examples for both, with SQL blocks to run in a Snowsight worksheet.

Use an existing database and `SNOWFLAKE_LEARNING_WH`. All lab objects belong to `EventInteractive`. Feature availability and the privileges needed to associate an event table are described before those steps.

## 1. Compare the two table types

| Type | Writer | Read pattern | Change model |
|---|---|---|---|
| Event | Instrumented handlers and Snowflake telemetry | Filter logs by time, severity, and object | Emit telemetry; apply a retention rule |
| Interactive | Source query or refresh service | Selective dashboard/API queries | Republish a static copy or refresh its source query |

The examples use order-validation logs and a small order dashboard dataset.

## 2. Set up the schema

Select an existing writable database and authorized role in Snowsight. Run setup once. Replace `EventInteractive` consistently with your team's schema name if needed.

```sql
CREATE SCHEMA EventInteractive;
USE SCHEMA EventInteractive;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;
SET LAB_DATABASE = CURRENT_DATABASE();
SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE(), CURRENT_ROLE();
```

The warehouse already exists. Keep the selected database and worksheet session throughout. `LAB_DATABASE` identifies that database only for telemetry configuration; no database is created.

## 3. Event table — observability data

An event table has a predefined schema aligned to OpenTelemetry. You do not declare columns. Important fields include `TIMESTAMP`, `START_TIMESTAMP`, `TRACE`, `RESOURCE_ATTRIBUTES`, `SCOPE`, `RECORD_TYPE`, `RECORD`, `RECORD_ATTRIBUTES`, `VALUE`, and `EXEMPLARS`. Semi-structured columns let one table hold logs, spans, metrics, and platform events.

The current account also has a Snowflake-provided default event table at `SNOWFLAKE.TELEMETRY.EVENTS`, exposed through `SNOWFLAKE.TELEMETRY.EVENTS_VIEW`. A custom table is useful for separate ownership, retention, and database-scoped telemetry. Source: [event table columns](https://docs.snowflake.com/en/developer-guide/logging-tracing/event-table-columns).

### Create and associate a custom event table

The setup role needs `CREATE EVENT TABLE`. Database-level event-table association requires Enterprise Edition and the applicable database ownership/event-table privileges. This setting routes telemetry for the selected database, not only this schema. Use a training database approved for this change.

First inspect and record the existing `EVENT_TABLE` setting so it can be restored after the lab:

```sql
SHOW PARAMETERS LIKE 'EVENT_TABLE' IN DATABASE IDENTIFIER($LAB_DATABASE);

CREATE EVENT TABLE ORDER_EVENTS
  DATA_RETENTION_TIME_IN_DAYS = 1
  COMMENT = 'Order-validation telemetry';

ALTER DATABASE IDENTIFIER($LAB_DATABASE)
  SET EVENT_TABLE = EventInteractive.ORDER_EVENTS;

SHOW EVENT TABLES IN SCHEMA EventInteractive;
DESCRIBE EVENT TABLE ORDER_EVENTS;
```

If the database association is not permitted, skip this custom-routing lab and query `SNOWFLAKE.TELEMETRY.EVENTS_VIEW` with a permitted role. Do not expect the unassociated custom table to receive the emitted logs.

[Event-table configuration](https://docs.snowflake.com/en/developer-guide/logging-tracing/event-table-setting-up)

### Create telemetry by running instrumented code

Applications do not normally `INSERT` hand-built rows into an event table. Handler code emits a log/trace/metric and Snowflake writes the correctly shaped event. This SQL stored procedure emits two log records.

```sql
CREATE PROCEDURE EMIT_ORDER_LOGS()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
  SYSTEM$LOG_INFO('ORDER_EVENT_DEMO: order validation started');
  SYSTEM$LOG_WARN('ORDER_EVENT_DEMO: sample warning for teaching');
  RETURN 'Two log calls completed';
END;
$$;

ALTER PROCEDURE EMIT_ORDER_LOGS() SET LOG_LEVEL = 'INFO';
CALL EMIT_ORDER_LOGS();
```

Telemetry ingestion is asynchronous and can take several seconds. Rerun the next `SELECT` if the new records are not visible immediately.

### Query, alter metadata, and delete by retention rule

`RECORD_TYPE = 'LOG'` identifies log rows; JSON path expressions extract severity and executable name. Event records are append-oriented evidence, so row `UPDATE` is not the correction model. Delete selected records or truncate the table according to an explicit retention policy.

```sql
SELECT TIMESTAMP AS EVENT_TIME_UTC,
       RESOURCE_ATTRIBUTES['snow.executable.name']::VARCHAR AS EXECUTABLE_NAME,
       RECORD['severity_text']::VARCHAR AS SEVERITY,
       VALUE::VARCHAR AS MESSAGE
FROM ORDER_EVENTS
WHERE RECORD_TYPE = 'LOG'
  AND VALUE::VARCHAR LIKE '%ORDER_EVENT_DEMO%'
ORDER BY TIMESTAMP DESC;

ALTER TABLE ORDER_EVENTS
  SET COMMENT = 'Order telemetry; one-day Time Travel; selective retention demo';

DELETE FROM ORDER_EVENTS
WHERE RECORD_TYPE = 'LOG'
  AND VALUE::VARCHAR LIKE '%ORDER_EVENT_DEMO:%'
  AND TIMESTAMP < DATEADD('minute', -30, CURRENT_TIMESTAMP());

-- Destructive retention option: removes every event row.
-- TRUNCATE TABLE ORDER_EVENTS;
```

Production event tables can grow quickly. Filter at the telemetry level, restrict sensitive values in logs, set a retention design, and monitor storage. Source: [working with event tables](https://docs.snowflake.com/en/developer-guide/logging-tracing/event-table-operations).

## 4. Interactive table — low-latency selective analytics

Interactive tables are generally available only in selected AWS, Azure, and GCP regions. Creation runs on a **standard warehouse**. The table must have `CLUSTER BY`, chosen from columns in the most latency-sensitive `WHERE` predicates. An interactive warehouse gives the intended cache and concurrency behavior, although a standard warehouse can query the table.

The supported SQL surface is narrower than for standard tables. `UPDATE`, `DELETE`, and ordinary `INSERT` are unsupported; `INSERT OVERWRITE` is the allowed DML. Interactive tables have Time Travel but no Fail-safe. Source: [interactive analytics](https://docs.snowflake.com/en/user-guide/interactive).

### Build a static interactive table

The source remains the mutable system of record. The interactive table is a serving copy optimized for customer/date filters.

```sql
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;
USE SCHEMA EventInteractive;

CREATE TABLE DASHBOARD_ORDERS (
  ORDER_ID INTEGER,
  CUSTOMER_ID INTEGER,
  ORDER_DATE DATE,
  REGION VARCHAR(20),
  STATUS VARCHAR(20),
  AMOUNT NUMBER(12,2)
);

INSERT INTO DASHBOARD_ORDERS VALUES
  (5001, 101, '2026-09-01', 'SOUTH', 'DELIVERED', 4500.00),
  (5002, 102, '2026-09-01', 'WEST',  'SHIPPED',   2200.00),
  (5003, 101, '2026-09-02', 'SOUTH', 'PLACED',   8500.00),
  (5004, 103, '2026-09-02', 'EAST',  'DELIVERED',3200.00);

CREATE INTERACTIVE TABLE ORDERS_INTERACTIVE
  CLUSTER BY (CUSTOMER_ID, ORDER_DATE)
  COMMENT = 'Static serving table for customer order lookups'
AS
SELECT * FROM DASHBOARD_ORDERS;

SELECT ORDER_ID, ORDER_DATE, STATUS, AMOUNT
FROM ORDERS_INTERACTIVE
WHERE CUSTOMER_ID = 101
  AND ORDER_DATE BETWEEN '2026-09-01' AND '2026-09-30'
ORDER BY ORDER_DATE, ORDER_ID;

SHOW INTERACTIVE TABLES LIKE 'ORDERS_INTERACTIVE';
```

### Replace the contents of a static interactive table

Change the standard source, then republish all rows with `INSERT OVERWRITE`. This preserves the interactive table object while replacing its contents.

```sql
UPDATE DASHBOARD_ORDERS SET STATUS = 'DELIVERED' WHERE ORDER_ID = 5002;
DELETE FROM DASHBOARD_ORDERS WHERE ORDER_ID = 5003;
INSERT INTO DASHBOARD_ORDERS VALUES
  (5005, 104, '2026-09-03', 'NORTH', 'PLACED', 6100.00);

INSERT OVERWRITE INTO ORDERS_INTERACTIVE
SELECT * FROM DASHBOARD_ORDERS;

SELECT * FROM ORDERS_INTERACTIVE ORDER BY ORDER_ID;
```

### Automatically refreshed interactive table

A target lag and refresh warehouse let Snowflake maintain this serving aggregate from the mutable order source. The existing standard warehouse is used for creation, refresh, and the functional query demonstration.

```sql
CREATE INTERACTIVE TABLE REGION_SALES_INTERACTIVE
  CLUSTER BY (REGION)
  TARGET_LAG = '5 minutes'
  WAREHOUSE = SNOWFLAKE_LEARNING_WH
  COMMENT = 'Automatically refreshed regional order aggregate'
AS
SELECT REGION, COUNT(*) AS ORDER_COUNT, SUM(AMOUNT) AS REVENUE
FROM DASHBOARD_ORDERS
GROUP BY REGION;

ALTER INTERACTIVE TABLE REGION_SALES_INTERACTIVE REFRESH;
SELECT * FROM REGION_SALES_INTERACTIVE WHERE REGION = 'SOUTH';
```

After the preceding order changes, South has one order and revenue 4500.00. This tiny lab does not benchmark interactive-serving latency. That requires an administrator-provided interactive warehouse and an appropriate workload; none is created here.

[Interactive analytics](https://docs.snowflake.com/en/user-guide/interactive)

## 5. Review

1. Which code emits the order logs, and why might they appear after a delay?
2. Why is an interactive table a serving copy of `DASHBOARD_ORDERS`?
3. What does `INSERT OVERWRITE` change in the static copy?
4. How is scheduled refresh different from republishing a static copy?

After the source changes, expect four orders: 5001, 5002, 5004, and 5005. Orders 5001 and 5002 are delivered. Log row counts depend on how often the procedure was called.

## 6. Optional cleanup

Restore the original database event-table association **before** dropping `ORDER_EVENTS`. If the database had no explicit association before this exercise, run the following. If it had one, replace the `UNSET` statement with `SET EVENT_TABLE =` followed by the original fully qualified event-table name you recorded. If you skipped association entirely, skip that statement.

```sql
USE SCHEMA EventInteractive;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;
ALTER DATABASE IDENTIFIER($LAB_DATABASE) UNSET EVENT_TABLE;

DROP TABLE IF EXISTS REGION_SALES_INTERACTIVE;
DROP TABLE IF EXISTS ORDERS_INTERACTIVE;
DROP TABLE IF EXISTS DASHBOARD_ORDERS;
DROP PROCEDURE IF EXISTS EMIT_ORDER_LOGS();
DROP TABLE IF EXISTS ORDER_EVENTS;
DROP SCHEMA IF EXISTS EventInteractive RESTRICT;
```

Run in the original database with the role that created these objects. The existing database and warehouse remain available.